# TSFGA full 실험 실행 노트북 (로컬 PC용)

이 노트북은 `ctfs` 프레임워크의 full 프리셋(합성 팹, 6공정, P=80, G=200)을 **로컬 PC에서** 돌리기 위한 것입니다.
위에서부터 순서대로 셀을 실행하세요. 실험은 별도 프로세스로 실행되므로 **VS Code를 닫거나 커널을 재시작해도 계속 돕니다.**
중간에 PC를 껐다면 4번 셀을 다시 실행하면 캐시된 결과를 건너뛰고 이어서 돕니다.

> GPU(RTX 5090)는 사용하지 않습니다. 실험은 랜덤포레스트+GA라 **CPU 코어 수**가 속도를 결정합니다.

## 1. 환경 확인

In [ ]:
import os, sys, platform, subprocess, time, json, glob
from pathlib import Path

ROOT = Path.cwd()                      # 이 노트북이 ctfs 폴더 안에 있어야 합니다
assert (ROOT / "run_experiments.py").exists(), "노트북을 ctfs 폴더 안에서 열어주세요 (run_experiments.py가 같은 폴더에 있어야 함)"
print("Python :", sys.version.split()[0], "|", platform.system(), platform.release())
print("CPU    :", os.cpu_count(), "logical cores")
try:
    import psutil
    print("RAM    :", round(psutil.virtual_memory().total / 1e9), "GB", "| physical cores:", psutil.cpu_count(logical=False))
except ImportError:
    print("RAM    : (psutil 미설치 - 2번 셀에서 설치됨)")

## 2. 패키지 설치 (처음 한 번만)

In [ ]:
%pip install -q -r requirements.txt psutil
import numpy, pandas, sklearn, scipy, matplotlib, seaborn, lightgbm, shap, pyarrow, joblib
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| sklearn", sklearn.__version__, "| lightgbm", lightgbm.__version__, "| shap", shap.__version__)

## 3. 2분짜리 동작 확인 (smoke test)

아주 작은 설정으로 파이프라인 전체(E1~E10 + 리포트)를 한 번 돌려 봅니다. 마지막 줄에 `report done`이 찍히면 정상입니다.

In [ ]:
cmd = [sys.executable, "run_experiments.py", "--preset", "quick", "--exp", "all", "--out", "results/smoke",
       "--set", "workers=2", "data.processes=2", "data.products_per_process=2", "data.ops_per_product=2",
       "data.records_per_combo=120", "ga.pop_size=6", "ga.n_gen=2", "seeds=0,1",
       "methods=ALL,FILTER,GA,TSFGA,NSGA2,mRMR,DDA,PFI-SBS,LASSO", "sensitivity.seeds=0",
       'sensitivity.grid={"lambda":[0.5,2.0],"pcc_threshold":[0.8,1.0]}',
       "temporal.seeds=0", "temporal.methods=ALL,TSFGA,DDA", "dataset2.processes=2", "dataset2.products_per_process=2",
       "dataset2.ops_per_product=2", "dataset2.records_per_combo=120", "dataset2_methods=ALL,TSFGA", "dataset2_seeds=0",
       "transfer.regressors=rf,dt", "interpret.n_shap=50"]
t0 = time.time()
r = subprocess.run(cmd, capture_output=True, text=True)
lines = [l for l in r.stdout.splitlines() if "=====" in l or "Traceback" in l or "Error" in l]
print("\n".join(lines[-6:]))
print(f"소요 {time.time()-t0:.0f}s | 정상 종료: {r.returncode == 0}")
if r.returncode != 0:
    print(r.stderr[-3000:])

## 4. full 실험 실행 (백그라운드)

아래 설정에서 `SEEDS`와 `WORKERS`만 필요에 따라 바꾸세요.

| 설정 | 의미 | 권장 |
|---|---|---|
| `WORKERS` | 동시에 돌릴 작업 수 | 물리 코어 수 (예: 16코어 CPU면 16). 논리 코어(스레드) 수로 올려도 10~20% 정도만 빨라집니다 |
| `SEEDS` | 시드 개수 | 5 (약 200 core-hour) 또는 10 (약 400 core-hour). 16코어 기준 5시드 ≈ 12~15시간, 10시드 ≈ 하루 이상 |
| `SENS_SEEDS` | 민감도 실험 시드 | 2 (5시드일 때) / 3 (10시드일 때) |

실행되는 동안 PC는 켜 두어야 하고(절전 모드 해제), VS Code는 닫아도 됩니다.

In [ ]:
WORKERS    = max(1, (os.cpu_count() or 2) // 2)   # 물리 코어 수 추정 (하이퍼스레딩 가정). 직접 지정 가능: WORKERS = 16
SEEDS      = 5                                     # 5 또는 10
SENS_SEEDS = 2 if SEEDS <= 5 else 3
OUT        = "results/full"

seeds = ",".join(str(i) for i in range(SEEDS))
cmd = [sys.executable, "-u", "run_experiments.py", "--preset", "full", "--exp", "all", "--out", OUT,
       "--set", f"workers={WORKERS}", f"seeds={seeds}", f"sensitivity.seeds={','.join(str(i) for i in range(SENS_SEEDS))}"]
log = open(f"{OUT.replace('/', '_')}.log", "a", encoding="utf-8")
kw = dict(stdout=log, stderr=subprocess.STDOUT, cwd=str(ROOT))
if platform.system() == "Windows":
    kw["creationflags"] = subprocess.CREATE_NEW_PROCESS_GROUP | subprocess.DETACHED_PROCESS
else:
    kw["start_new_session"] = True
proc = subprocess.Popen(cmd, **kw)
open("full_run.pid", "w").write(str(proc.pid))
print("시작됨. PID =", proc.pid, "| 로그:", log.name)
print("명령:", " ".join(cmd))

## 5. 진행 상황 확인 (아무 때나 실행)

main 단계는 총 `6공정 × 14방법 × SEEDS` 개의 작업이며, 캐시 파일 수로 진행률과 남은 시간을 추정합니다.

In [ ]:
def progress(out="results/full", seeds=None):
    log_path = f"{out.replace('/', '_')}.log"
    lines = open(log_path, encoding="utf-8", errors="ignore").read().splitlines() if os.path.exists(log_path) else []
    stages = [l for l in lines if "=====" in l]
    errors = [l for l in lines if "Traceback" in l or "Error" in l]
    print("현재 단계:", stages[-1] if stages else "(로그 없음)")
    if errors: print("!! 오류:", errors[-1])
    cfg = json.load(open(f"{out}/config.json")) if os.path.exists(f"{out}/config.json") else None
    if cfg:
        n_total = 6 * len(cfg["methods"]) * len(cfg["seeds"])
        done = len(glob.glob(f"{out}/main/cache/*.json"))
        heavy = sum(1 for f in glob.glob(f"{out}/main/cache/*.json") if any(m in f for m in ["_GA_", "_TSFGA_", "_BPSO_", "_BGWO_", "_NSGA2_"]))
        print(f"main: {done}/{n_total} 작업 완료 (메타휴리스틱 {heavy}/{5*6*len(cfg['seeds'])})")
        for stage, sub in [("sensitivity", "E4_sensitivity/cache"), ("temporal", "E7_temporal/split_*/cache"), ("dataset2", "E9_dataset2/cache")]:
            n = len(glob.glob(f"{out}/{sub}/*.json"))
            if n: print(f"{stage}: {n} 작업 완료")
    tail = [l for l in lines if l[:2].isdigit()][-3:]
    print("최근 로그:"); print("\n".join(tail))
    alive = False
    if os.path.exists("full_run.pid"):
        pid = int(open("full_run.pid").read())
        try:
            import psutil; alive = psutil.pid_exists(pid)
        except ImportError: pass
    print("프로세스 실행 중:", alive)
progress(OUT if 'OUT' in dir() else "results/full")

## 6. (필요시) 중단하기

실험을 멈추려면 아래 셀을 실행하세요. 캐시는 남아 있으므로 4번 셀을 다시 실행하면 이어서 돕니다.

In [ ]:
import signal
pid = int(open("full_run.pid").read())
try:
    import psutil
    p = psutil.Process(pid)
    for c in p.children(recursive=True): c.kill()
    p.kill(); print("중단됨:", pid)
except Exception as e:
    print("이미 종료되었거나 찾을 수 없음:", e)

## 7. 완료 후: 원고용 데이터 생성 + 전달용 zip 만들기

로그에 `===== report done` 이 찍힌 뒤 실행하세요. `results_full_for_manuscript.zip` 한 파일을 보내주시면 원고를 갱신하겠습니다.

In [ ]:
import shutil, zipfile
OUT = OUT if 'OUT' in dir() else "results/full"
r = subprocess.run([sys.executable, "tools/build_manuscript_data.py", OUT, f"{OUT}/manuscript_data.json"], capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-1500:])
zip_path = "results_full_for_manuscript.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in Path(OUT).rglob("*"):
        if f.is_file() and "cache" not in f.parts and f.suffix != ".parquet":
            z.write(f, f.relative_to(Path(OUT).parent))
    lp = f"{OUT.replace('/', '_')}.log"
    if os.path.exists(lp): z.write(lp)
print("만들어짐:", zip_path, round(os.path.getsize(zip_path)/1e6, 1), "MB")
print("REPORT:", (Path(OUT) / "REPORT.md").exists())